In [25]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from scipy.spatial.distance import cdist

In [26]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
candidates_df = pd.read_csv('candidates.csv')

def parse_coords_to_matrix(series):
    return np.vstack(
        series.apply(lambda x: np.fromstring(str(x).strip('|'), dtype=float, sep='|')).values
    )

# 2. X Train Coords & Y Train Text
X_train_coords = parse_coords_to_matrix(train_df['coords'])
y_train_text = train_df['text'].values

# 3. Test Coords (Aliniere strictă după datapointID sortat)
test_dp_ids = sorted(test_df['datapointID'].unique())
X_test_coords_list = []

for dp_id in test_dp_ids:
    group = test_df[test_df['datapointID'] == dp_id]
    coords_mat = parse_coords_to_matrix(group['coords'])
    X_test_coords_list.append(coords_mat.mean(axis=0))

X_test_coords = np.array(X_test_coords_list)

In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_coords)
X_test_scaled = scaler.transform(X_test_coords)

# 5. Vectorizare Text
vectorizer = TfidfVectorizer(max_features=1000, stop_words='english')
Y_train_tfidf = vectorizer.fit_transform(y_train_text).toarray()
Y_candidates_tfidf = vectorizer.transform(candidates_df['text'].values).toarray()

In [28]:
model = Ridge(alpha=10.0) # alpha mai mare previne overfit-ul
model.fit(X_train_scaled, Y_train_tfidf)

X_test_pred = model.predict(X_test_scaled)

# 7. Potrivire Cosinus
dists = cdist(X_test_pred, Y_candidates_tfidf, metric='cosine')

candidate_text_ids = candidates_df['text_id'].values
best_answers = []

for i in range(len(test_dp_ids)):
    best_idx = np.argmin(dists[i])
    # Preluăm valoarea int reală din coloana text_id
    best_answers.append(int(candidate_text_ids[best_idx]))

In [29]:
sub = pd.DataFrame({
    'subtaskID': 1,
    'datapointID': test_dp_ids,
    'answer': best_answers
})

sub.to_csv('submission.csv', index=False)